## Compiler Optimization

In [1]:
import torch
import os
from Simulator.simulator import TOGSimulator

fusion_config = "/workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_timing_only.yml"
nonfusion_config = "/workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_no_compiler_optimization.yml"

### GeMM + ReLU fusion (Default)

In [ ]:
os.environ['TORCHSIM_DUMP_PATH']=os.path.join(os.getcwd(), "fused")

device = torch.device("npu:0")

input = torch.randn(1024, 1024).to(device=device)
weight = torch.randn(1024, 1024).to(device=device)

def gemm_relu(a, b):
    return torch.relu(torch.matmul(a, b))

opt_fn = torch.compile(dynamic=False)(gemm_relu)

with TOGSimulator(config_path=fusion_config):
    npu_output = opt_fn(input, weight)

### Disabling fusion

In [ ]:
torch._dynamo.reset()
os.environ['TORCHSIM_DUMP_PATH']=os.path.join(os.getcwd(), "non_fused")

input = torch.randn(1024, 1024).to(device=device)
weight = torch.randn(1024, 1024).to(device=device)

def gemm_relu(a, b):
    return torch.relu(torch.matmul(a, b))

opt_fn = torch.compile(dynamic=False)(gemm_relu)

with TOGSimulator(config_path=nonfusion_config):
    npu_output = opt_fn(input, weight)